# Appendix Tables — WaveletFlow Paper

## 0. Imports & Global Settings

In [1]:
import glob
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

rc = {
    "axes.axisbelow":      True,
    "axes.linewidth":      1.0,
    "axes.labelsize":      "x-small",
    "axes.titlesize":      "x-small",
    "figure.dpi":          150,
    "figure.figsize":      (3.2, 3.2),
    "font.size":           12.0,
    "legend.fontsize":     "xx-small",
    "lines.linewidth":     1.0,
    "savefig.transparent": True,
    "xtick.labelsize":     "xx-small",
    "xtick.major.width":   1.0,
    "ytick.labelsize":     "xx-small",
    "ytick.major.width":   1.0,
    "font.family":         ["serif"],
    "text.usetex":         True,
    "text.latex.preamble": r"\usepackage{amsfonts}",
}
plt.rcParams.update(rc)

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

## 1. Model Registry & Derived Sets

In [2]:
def rgb(r, g, b): return f"#{r:02x}{g:02x}{b:02x}"

# Each entry: label, color, dash, lw, group in {wfm-1scale, wfm-3scale, generative, deterministic}
MODEL_REGISTRY = {
    # -- WFM 1-scale variants --------------------------------------------------
    "WaveletFlow_1scale_haar_v4.0": dict(label="WFM-1 haar", color=rgb( 98,124,107), dash=(1,0), lw=2.5, group="wfm-1scale"),
    "WaveletFlow_1scale_db2_v4.0":  dict(label="WFM-1 db2",  color=rgb( 98,124,107), dash=(1,0), lw=2.5, group="wfm-1scale"),
    "WaveletFlow_1scale_db4_v4.0":  dict(label="WFM-1 db4",  color=rgb( 98,124,107), dash=(1,0), lw=2.5, group="wfm-1scale"),
    "WaveletFlow_1scale_db6_v4.0":  dict(label="WFM-1 db6",  color=rgb( 98,124,107), dash=(1,0), lw=2.5, group="wfm-1scale"),
    # -- WFM 3-scale variants --------------------------------------------------
    "WaveletFlow_3scales_haar_v4.0": dict(label="WFM-3 haar", color=rgb(117,176,138), dash=(1,0), lw=2.5, group="wfm-3scale"),
    "WaveletFlow_3scales_db2_v4.0":  dict(label="WFM-3 db2",  color=rgb(117,176,138), dash=(1,0), lw=2.5, group="wfm-3scale"),
    "WaveletFlow_3scales_db4_v4.0":  dict(label="WFM-3 db4",  color=rgb(117,176,138), dash=(1,0), lw=2.5, group="wfm-3scale"),
    "WaveletFlow_3scales_db6_v4.0":  dict(label="WFM-3 db6",  color=rgb(117,176,138), dash=(1,0), lw=2.5, group="wfm-3scale"),
    # -- Generative baselines --------------------------------------------------
    "FM_data_space_v4.0":       dict(label=r"FM$_\text{pixel}$",   color=rgb(176,176,176), dash=(1,0), lw=1.8, group="generative"),
    "FourierFlow_32modes_v4.0": dict(label=r"FM$_\text{fourier}$", color=rgb(143, 94,106), dash=(1,0), lw=1.8, group="generative"),
    "WDNO":                     dict(label="WDNO",                  color=rgb(191,134,148), dash=(1,0), lw=1.8, group="generative"),
    # -- Deterministic baselines -----------------------------------------------
    "WNO_3scales_db2": dict(label="WNO",  color=rgb(176,136, 84), dash=(1,0), lw=1.8, group="deterministic"),
    "FNO":             dict(label="FNO",  color=rgb(208,169,110), dash=(1,0), lw=1.8, group="deterministic"),
    "TFNO":            dict(label="TFNO", color=rgb(236,227,163), dash=(1,0), lw=1.8, group="deterministic"),
}

all_models = list(MODEL_REGISTRY)
WFM_1scale = [m for m in all_models if MODEL_REGISTRY[m]["group"] == "wfm-1scale"]
WFM_3scale = [m for m in all_models if MODEL_REGISTRY[m]["group"] == "wfm-3scale"]
WFM_all    = WFM_1scale + WFM_3scale
gen_models = [m for m in all_models if MODEL_REGISTRY[m]["group"] == "generative"]
det_models = [m for m in all_models if MODEL_REGISTRY[m]["group"] == "deterministic"]

# Canonical table ordering: WFM-1scale -> WFM-3scale -> generative -> deterministic
MODEL_ORDER = WFM_1scale + WFM_3scale + gen_models + det_models

labels     = {m: MODEL_REGISTRY[m]["label"] for m in all_models}
palette    = {m: MODEL_REGISTRY[m]["color"] for m in all_models}
dashes     = {m: MODEL_REGISTRY[m]["dash"]  for m in all_models}
linewidths = {m: MODEL_REGISTRY[m]["lw"]    for m in all_models}

GT_LABEL = "Ground-truth"
palette[GT_LABEL] = "#222222"; dashes[GT_LABEL] = (1, 0)
labels[GT_LABEL]  = GT_LABEL;  linewidths[GT_LABEL] = 1.8

def model_group(m):
    if m in WFM_1scale: return "wfm-1scale"
    if m in WFM_3scale: return "wfm-3scale"
    if m in gen_models: return "generative"
    return "deterministic"

## 2. PDE Metadata

In [3]:
PDEs = ["turbulent_radiative_layer_2D", "rayleigh_benard", "active_matter"]

PDE_LABELS = {
    "turbulent_radiative_layer_2D": "Turbulence",
    "rayleigh_benard":              r"Rayleigh--B\'{e}nard",
    "active_matter":                "Active Matter",
}

# Physical parameters varied in ablation studies
PDE_CONSTANTS = {
    "turbulent_radiative_layer_2D": ["tcool"],
    "rayleigh_benard":              ["Rayleigh", "Prandtl"],
    "active_matter":                ["zeta", "alpha"],
}

# Raw horizon strings produced by the step->horizon mapping (per PDE)
HORIZON_BINS_RAW = {
    "turbulent_radiative_layer_2D": ["01:32", "33:64", "65:97"],
    "rayleigh_benard":              ["01:32", "33:64", "65:196"],
    "active_matter":                ["01:16", "17:32", "33:77"],
}

# Generic column labels for unified multi-PDE tables (Short / Mid / Long)
GENERIC_HORIZONS = ["Short", "Mid", "Long"]
HORIZON_TO_GENERIC = {
    pde: dict(zip(raw, GENERIC_HORIZONS))
    for pde, raw in HORIZON_BINS_RAW.items()
}

# Spectral frequency bands
BANDS     = ["Low", "Mid", "High"]
BAND_COLS = {"rmse_c_low": "Low", "rmse_c_mid": "Mid", "rmse_c_high": "High"}

# Horizon caption note (reused in LaTeX exports)
HZ_CAPTION_NOTE = (
    r"Short/Mid/Long $\equiv$ [1\!:\!32]/[33\!:\!64]/[65\!:\!97] (Turbulence), "
    r"[1\!:\!32]/[33\!:\!64]/[65\!:\!196] (Rayleigh--B\'enard), "
    r"[1\!:\!16]/[17\!:\!32]/[33\!:\!77] (Active Matter)."
)

## 3. Data Loading

In [4]:
LOAD_STATS = True
LOAD_PSD   = False


def enrich(df, stem, model_name, dataset_name):
    # Preserve GT rows already labelled in the CSV
    parts     = stem.split("_")
    nsampling = next((p.replace("nsampling", "") for p in parts if p.startswith("nsampling")), None)
    nens      = next((p.replace("nens",      "") for p in parts if p.startswith("nens")),      None)
    if "model" in df.columns:
        df.loc[df["model"] != GT_LABEL, "model"] = model_name
    else:
        df["model"] = model_name
    df["source"]    = stem
    df["dataset"]   = dataset_name
    df["pde"]       = dataset_name
    df["nsampling"] = int(nsampling) if nsampling else None
    df["nens"]      = int(nens)      if nens      else None
    return df


DEDUP_EXCLUDE = {"source", "nsampling", "nens"}

dfs_stats, dfs_psd = [], []
for pde in PDEs:
    print(f"\n{'='*60}\nLoading: {pde}\n{'='*60}")
    for model in all_models:
        for suffix, target, flag in [
            ("stats", dfs_stats, LOAD_STATS),
            ("psd",   dfs_psd,   LOAD_PSD),
        ]:
            if not flag:
                continue
            pattern = f"{PROJECT_ROOT}/experiments/{pde}/metrics/{model}*{suffix}.csv"
            matched = sorted(glob.glob(pattern))
            if not matched:
                print(f"  [MISS {suffix}] {model}")
            for fpath in matched:
                try:
                    df = pd.read_csv(fpath)
                    target.append(enrich(df, Path(fpath).stem, model, pde))
                    print(f"  [OK   {suffix}] {Path(fpath).name}")
                except Exception as exc:
                    print(f"  [ERR] {fpath}: {exc}")


def _concat_dedup(dfs):
    if not dfs:
        return pd.DataFrame()
    full = pd.concat(dfs, ignore_index=True)
    return full.drop_duplicates(
        subset=[c for c in full.columns if c not in DEDUP_EXCLUDE]
    )


stats_df = _concat_dedup(dfs_stats)
psd_df   = _concat_dedup(dfs_psd)
print(f"\nstats_df : {stats_df.shape}   cols: {list(stats_df.columns)}")
print(f"psd_df   : {psd_df.shape}   cols: {list(psd_df.columns)}")


Loading: turbulent_radiative_layer_2D
  [OK   stats] WaveletFlow_1scale_haar_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_1scale_db2_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_1scale_db4_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_1scale_db6_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_3scales_haar_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_3scales_db2_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_3scales_db4_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] WaveletFlow_3scales_db6_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] FM_data_space_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_stats.csv
  [OK   stats] FourierFlow_32modes_v4.0_nsampling50_nens8_turbul

## 4. Feature Engineering

In [5]:
AGG_MODE = "mean"   # "mean" | "median"

def aggregate(x):
    return np.nanmean(x) if AGG_MODE == "mean" else np.nanmedian(x)


def _horizon_fn(bins):
    # Build step -> horizon_label function from ordered (threshold, label) pairs
    def fn(t):
        for thresh, label in bins:
            if t <= thresh:
                return label
        return bins[-1][1]
    return fn


HORIZON_FN = {
    "turbulent_radiative_layer_2D": _horizon_fn([(32,"01:32"), (64,"33:64"), (97, "65:97")]),
    "rayleigh_benard":              _horizon_fn([(32,"01:32"), (64,"33:64"), (196,"65:196")]),
    "active_matter":                _horizon_fn([(16,"01:16"), (32,"17:32"), (77, "33:77")]),
}

if LOAD_STATS and not stats_df.empty:
    stats_df = stats_df.copy()
    stats_df["step"] += 1
    stats_df["horizon"] = pd.NA
    for pde, fn in HORIZON_FN.items():
        mask = stats_df["pde"] == pde
        stats_df.loc[mask, "horizon"] = stats_df.loc[mask, "step"].apply(fn)
    stats_df["model_group"] = stats_df["model"].apply(model_group)
    if {"spread", "rmse"}.issubset(stats_df.columns):
        stats_df["spread_skill"] = (stats_df["spread"] + 1e-3) / (stats_df["rmse"] + 1e-3)

if LOAD_PSD and not psd_df.empty:
    psd_df = psd_df.copy()
    psd_df["step"] += 1
    psd_df["horizon"] = pd.NA
    for pde, fn in HORIZON_FN.items():
        mask = psd_df["pde"] == pde
        psd_df.loc[mask, "horizon"] = psd_df.loc[mask, "step"].apply(fn)
    psd_df["model_group"] = psd_df["model"].apply(model_group)

## 5. Styling & LaTeX Helpers

In [6]:
# ============================================================
# Jupyter display helpers
# ============================================================

def highlight_min(df: pd.DataFrame) -> pd.DataFrame:
    # Bold minimum, underline 2nd minimum, per column
    out = pd.DataFrame("", index=df.index, columns=df.columns)
    for col in df.columns:
        vals = df[col].dropna().sort_values()
        if len(vals) >= 1: out.loc[vals.index[0], col] = "font-weight: bold"
        if len(vals) >= 2: out.loc[vals.index[1], col] = "text-decoration: underline"
    return out


def highlight_min_pde(df: pd.DataFrame) -> pd.DataFrame:
    # Bold minimum, underline 2nd minimum, per column, WITHIN each PDE group
    out = pd.DataFrame("", index=df.index, columns=df.columns)
    for pde in df.index.get_level_values(0).unique():
        mask = df.index.get_level_values(0) == pde
        sub  = df.loc[mask]
        for col in df.columns:
            vals = sub[col].dropna().sort_values()
            if len(vals) >= 1: out.loc[vals.index[0], col] = "font-weight: bold"
            if len(vals) >= 2: out.loc[vals.index[1], col] = "text-decoration: underline"
    return out


# ============================================================
# LaTeX helpers
# ============================================================

def _fmt_cell(val: float, rank: int) -> str:
    s = f"{val:.3f}"
    if rank == 1: return r"\textbf{" + s + "}"
    if rank == 2: return r"\underline{" + s + "}"
    return s


def to_latex_multiindex(
    df: pd.DataFrame,
    caption: str,
    label: str,
    minimize: bool = True,
    header_lines: list = None,
) -> str:
    # Export a (PDE, Model) 2-level MultiIndex DataFrame to a LaTeX booktabs table.
    #
    # Layout:
    #   - PDE groups are delimited by \midrule.
    #   - The PDE cell spans its group rows via \multirow + \rotatebox{90}.
    #   - Bold  = best  per column *within* the PDE block.
    #   - Underline = 2nd best per column *within* the PDE block.
    assert isinstance(df.index, pd.MultiIndex) and df.index.nlevels == 2

    # Per-PDE-block column ranks:  {(pde_label, model_label, col): rank}
    ranks = {}
    for pde in df.index.get_level_values(0).unique():
        mask = df.index.get_level_values(0) == pde
        sub  = df.loc[mask]
        for col in df.columns:
            sorted_idx = sub[col].dropna().sort_values(ascending=minimize).index.tolist()
            for rank, idx in enumerate(sorted_idx, 1):
                ranks[(pde, idx[1], col)] = rank   # idx = (pde_label, model_label)

    pde_counts = {
        pde: int((df.index.get_level_values(0) == pde).sum())
        for pde in df.index.get_level_values(0).unique()
    }

    n_cols   = len(df.columns)
    col_spec = "ll" + "c" * n_cols

    if header_lines is None:
        header_lines = [
            "PDE & Model & " + " & ".join(str(c) for c in df.columns) + r" \\"
        ]

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\small",
        r"\begin{tabular}{" + col_spec + "}",
        r"\toprule",
        *header_lines,
        r"\midrule",
    ]

    prev_pde = None
    for (pde, model), row in df.iterrows():
        if prev_pde is not None and pde != prev_pde:
            lines.append(r"\midrule")
        pde_cell = (
            r"\multirow{" + str(pde_counts[pde]) + r"}{*}{\rotatebox{90}{" + pde + "}}"
            if pde != prev_pde else ""
        )
        prev_pde = pde
        cells = [pde_cell, model]
        for col in df.columns:
            val  = row[col]
            rank = ranks.get((pde, model, col), 99)
            cells.append("--" if pd.isna(val) else _fmt_cell(val, rank))
        lines.append(" & ".join(cells) + r" \\")

    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\end{table}",
    ]
    return "\n".join(lines)


def to_latex_simple(
    df: pd.DataFrame,
    caption: str,
    label: str,
    minimize: bool = True,
    header_lines: list = None,
) -> str:
    # Export a flat-index DataFrame to a LaTeX booktabs table.
    # Bold = best, underline = 2nd best, per column.
    ranks = {}
    for col in df.columns:
        sorted_idx = df[col].dropna().sort_values(ascending=minimize).index.tolist()
        ranks[col] = {m: i + 1 for i, m in enumerate(sorted_idx)}

    n_cols = len(df.columns)

    if header_lines is None:
        header_lines = [
            "Model & " + " & ".join(str(c) for c in df.columns) + r" \\"
        ]

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\small",
        r"\begin{tabular}{l" + "c" * n_cols + "}",
        r"\toprule",
        *header_lines,
        r"\midrule",
    ]
    for model, row in df.iterrows():
        cells = [str(model)]
        for col in df.columns:
            val  = row[col]
            rank = ranks[col].get(model, 99)
            cells.append("--" if pd.isna(val) else _fmt_cell(val, rank))
        lines.append(" & ".join(cells) + r" \\")
    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\end{table}",
    ]
    return "\n".join(lines)

## 6. Table Builders

In [7]:
# ============================================================
# Main appendix tables
# ============================================================

def build_main_table(
    metric: str,
    models: list = None,
    pdes: list = None,
) -> pd.DataFrame:
    # Build a unified multi-PDE appendix table.
    #
    # Rows : 2-level MultiIndex  (PDE display label,  model display label)
    #        ordered by MODEL_ORDER within each PDE block.
    # Cols : Short / Mid / Long  (generic horizon labels, consistent across PDEs).
    #
    # metric : stats_df column to aggregate (e.g. "vrmse", "crps").
    # models : ordered subset of model keys; defaults to MODEL_ORDER.
    # pdes   : ordered subset of PDE keys;   defaults to PDEs.
    if models is None: models = MODEL_ORDER
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "horizon", metric]]
            .groupby(["model", "horizon"])[metric]
            .agg(aggregate).round(3).reset_index()
        )
        sub["horizon"] = sub["horizon"].map(HORIZON_TO_GENERIC[pde])
        sub = sub.pivot(index="model", columns="horizon", values=metric)
        sub = sub.reindex(
            index=[m for m in models if m in sub.index],
            columns=[h for h in GENERIC_HORIZONS if h in sub.columns],
        )
        sub.index = pd.MultiIndex.from_tuples(
            [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
            names=["PDE", "Model"],
        )
        frames.append(sub)
    return pd.concat(frames)


def build_spectral_table(
    models: list = None,
    pdes: list = None,
) -> pd.DataFrame:
    # Multi-PDE spectral coherence RMSE table (averaged over horizons).
    # Rows : (PDE, model) MultiIndex
    # Cols : Low / Mid / High
    if models is None: models = MODEL_ORDER
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "horizon"] + list(BAND_COLS.keys())]
            .groupby(["model", "horizon"])[list(BAND_COLS.keys())]
            .agg(aggregate).reset_index()
            .rename(columns=BAND_COLS)
            .groupby("model")[BANDS]
            .mean().round(3)
        )
        sub = sub.reindex([m for m in models if m in sub.index])
        sub.index = pd.MultiIndex.from_tuples(
            [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
            names=["PDE", "Model"],
        )
        frames.append(sub)
    return pd.concat(frames)


def build_spectral_horizon_table(
    models: list = None,
    pdes: list = None,
) -> pd.DataFrame:
    # Per-PDE spectral table broken down by horizon AND band.
    # Rows : (PDE, model) MultiIndex
    # Cols : (band, horizon) MultiIndex
    if models is None: models = MODEL_ORDER
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "horizon"] + list(BAND_COLS.keys())]
            .melt(id_vars=["model", "horizon"],
                  value_vars=list(BAND_COLS.keys()),
                  var_name="band_key", value_name="srmse")
            .assign(band=lambda x: x["band_key"].map(BAND_COLS),
                    horizon=lambda x: x["horizon"].map(HORIZON_TO_GENERIC[pde]))
            .groupby(["model", "band", "horizon"])["srmse"]
            .agg(aggregate).round(3).reset_index()
            .pivot(index="model", columns=["band", "horizon"], values="srmse")
            .reindex(columns=BANDS, level="band")
            .reindex(columns=GENERIC_HORIZONS, level="horizon")
        )
        sub = sub.reindex([m for m in models if m in sub.index])
        sub.index = pd.MultiIndex.from_tuples(
            [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
            names=["PDE", "Model"],
        )
        frames.append(sub)
    return pd.concat(frames)


def build_w1_table(
    models: list = None,
    pdes: list = None,
) -> pd.DataFrame:
    # Wasserstein-1 table: (PDE, model) rows x single W1 column.
    if models is None: models = WFM_all + gen_models
    if pdes   is None: pdes   = PDEs

    frames = []
    for pde in pdes:
        sub = (
            stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
            [["model", "w1"]]
            .groupby("model")["w1"]
            .agg(aggregate).round(3)
            .reset_index().set_index("model")
        )
        sub = sub.reindex([m for m in models if m in sub.index])
        sub.index = pd.MultiIndex.from_tuples(
            [(PDE_LABELS[pde], labels.get(m, m)) for m in sub.index],
            names=["PDE", "Model"],
        )
        sub.columns = [r"$W_1$"]
        frames.append(sub)
    return pd.concat(frames)


# ============================================================
# Ablation tables
# ============================================================

def build_ablation_table(
    metric: str,
    pde: str,
    param: str,
    models: list = None,
    param_filter=None,
) -> pd.DataFrame:
    # Ablation table: model rows x param-value columns.
    #
    # metric       : "vrmse", "crps", etc.
    # pde          : PDE key string.
    # param        : physical parameter column (e.g. "tcool", "Rayleigh", "alpha").
    # models       : models to show; defaults to WFM_all + gen_models.
    # param_filter : optional callable applied to param values,
    #                e.g. ``lambda v: v <= 0.03`` to restrict tcool range.
    if models is None: models = WFM_all + gen_models

    sub = (
        stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
        [["model", param, metric]]
        .dropna(subset=[param, metric])
    )
    if param_filter is not None:
        sub = sub[sub[param].apply(param_filter)]

    sub = (
        sub.groupby(["model", param])[metric]
        .agg(aggregate).round(3).reset_index()
        .pivot(index="model", columns=param, values=metric)
    )
    sub = sub.reindex([m for m in models if m in sub.index])
    sub.index = [labels.get(m, m) for m in sub.index]
    sub.index.name  = "Model"
    sub.columns.name = param
    return sub


def build_spectral_ablation_table(
    pde: str,
    param: str,
    models: list = None,
    param_filter=None,
) -> pd.DataFrame:
    # Spectral ablation table: model rows x (band, param_value) columns.
    if models is None: models = WFM_all + gen_models

    sub = (
        stats_df[(stats_df["pde"] == pde) & (stats_df["model"].isin(models))]
        [["model", param] + list(BAND_COLS.keys())]
        .dropna(subset=[param])
    )
    if param_filter is not None:
        sub = sub[sub[param].apply(param_filter)]

    sub = (
        sub.melt(id_vars=["model", param],
                 value_vars=list(BAND_COLS.keys()),
                 var_name="band_key", value_name="srmse")
        .assign(band=lambda x: x["band_key"].map(BAND_COLS))
        .groupby(["model", "band", param])["srmse"]
        .agg(aggregate).round(3).reset_index()
        .pivot(index="model", columns=["band", param], values="srmse")
        .reindex(columns=BANDS, level="band")
    )
    sub = sub.reindex([m for m in models if m in sub.index])
    sub.index = [labels.get(m, m) for m in sub.index]
    sub.index.name = "Model"
    return sub

---
# A. Main Appendix Tables

**Row structure**: 2-level MultiIndex `(PDE, Model)`.
WFM-1scale variants first (haar → db2 → db4 → db6), then WFM-3scale, then generative baselines, then deterministic baselines.

**Column structure**: Short / Mid / Long horizons (main tables) or Low / Mid / High frequency bands (spectral).

**Highlighting**: **bold** = best, <u>underline</u> = 2nd best, ranked *within each PDE block*.

## A.1  VRMSE — All Models

In [8]:
t_vrmse = build_main_table(metric="vrmse", models=MODEL_ORDER)
t_vrmse.style.apply(highlight_min_pde, axis=None).format("{:.3f}", na_rep="--")

/var/folders/ch/y8sfh_4n0879ll031cvbsv380000gn/T/ipykernel_14787/3832332138.py:4: RuntimeWarning: Mean of empty slice
  return np.nanmean(x) if AGG_MODE == "mean" else np.nanmedian(x)


In [9]:
print(to_latex_multiindex(
    t_vrmse,
    caption=(
        r"VRMSE $\downarrow$ per PDE and rollout horizon. "
        + HZ_CAPTION_NOTE
        + r" \textbf{Bold}: best; \underline{underline}: 2nd best within each PDE block."
    ),
    label="tab:appendix_vrmse",
    header_lines=[r"PDE & Model & Short & Mid & Long \\"],
))

\begin{table}[t]
\centering
\small
\begin{tabular}{llccc}
\toprule
PDE & Model & Short & Mid & Long \\
\midrule
\multirow{14}{*}{\rotatebox{90}{Turbulence}} & WFM-1 haar & \underline{0.545} & 0.765 & 0.782 \\
 & WFM-1 db2 & 0.553 & 0.752 & 0.856 \\
 & WFM-1 db4 & 0.556 & \underline{0.731} & \textbf{0.776} \\
 & WFM-1 db6 & \textbf{0.544} & \textbf{0.705} & 0.787 \\
 & WFM-3 haar & 0.557 & 0.859 & 0.810 \\
 & WFM-3 db2 & 0.635 & 0.892 & 0.934 \\
 & WFM-3 db4 & 0.578 & 0.763 & 0.817 \\
 & WFM-3 db6 & 0.582 & 0.893 & 0.821 \\
 & FM$_\text{pixel}$ & 0.576 & 0.920 & 0.865 \\
 & FM$_\text{fourier}$ & 0.602 & 0.740 & \underline{0.779} \\
 & WDNO & 1.517 & 1.835 & 1.953 \\
 & WNO & inf & inf & -- \\
 & FNO & 2.952 & 4.978 & 4.349 \\
 & TFNO & 2.300 & 6.534 & 6.387 \\
\midrule
\multirow{14}{*}{\rotatebox{90}{Rayleigh--B\'{e}nard}} & WFM-1 haar & 1.778 & 1.985 & 0.811 \\
 & WFM-1 db2 & 1.326 & 1.893 & 0.797 \\
 & WFM-1 db4 & 1.732 & 2.362 & 0.800 \\
 & WFM-1 db6 & 1.463 & 1.998 & 0.980 \\
 & WFM

## A.2  CRPS — WFM + Generative Models

In [10]:
t_crps = build_main_table(metric="crps", models=WFM_all + gen_models)
t_crps.style.apply(highlight_min_pde, axis=None).format("{:.3f}", na_rep="--")

In [11]:
print(to_latex_multiindex(
    t_crps,
    caption=(
        r"CRPS $\downarrow$ for generative and WFM models. " + HZ_CAPTION_NOTE
    ),
    label="tab:appendix_crps",
    header_lines=[r"PDE & Model & Short & Mid & Long \\"],
))

\begin{table}[t]
\centering
\small
\begin{tabular}{llccc}
\toprule
PDE & Model & Short & Mid & Long \\
\midrule
\multirow{11}{*}{\rotatebox{90}{Turbulence}} & WFM-1 haar & 0.503 & 1.049 & 1.193 \\
 & WFM-1 db2 & 0.449 & 1.144 & 1.427 \\
 & WFM-1 db4 & 0.532 & 1.079 & 1.228 \\
 & WFM-1 db6 & 0.451 & \underline{1.036} & 1.197 \\
 & WFM-3 haar & \textbf{0.415} & 1.073 & \underline{1.173} \\
 & WFM-3 db2 & 0.450 & 1.143 & 1.227 \\
 & WFM-3 db4 & 0.426 & \textbf{0.994} & \textbf{1.124} \\
 & WFM-3 db6 & 0.443 & 1.041 & 1.180 \\
 & FM$_\text{pixel}$ & \underline{0.422} & 1.093 & 1.236 \\
 & FM$_\text{fourier}$ & 0.854 & 1.391 & 1.528 \\
 & WDNO & 2.460 & 3.372 & 4.285 \\
\midrule
\multirow{11}{*}{\rotatebox{90}{Rayleigh--B\'{e}nard}} & WFM-1 haar & 0.005 & 0.036 & 0.078 \\
 & WFM-1 db2 & 0.005 & 0.036 & \underline{0.072} \\
 & WFM-1 db4 & 0.007 & 0.041 & 0.072 \\
 & WFM-1 db6 & 0.005 & 0.040 & 0.091 \\
 & WFM-3 haar & \underline{0.004} & 0.036 & 0.077 \\
 & WFM-3 db2 & \textbf{0.003} & \text

## A.3  Spectral Coherence RMSE — WFM + Generative Models

Values averaged over all rollout horizons within each PDE.

In [12]:
t_spectral = build_spectral_table(models=WFM_all + gen_models)
t_spectral.style.apply(highlight_min_pde, axis=None).format("{:.3f}", na_rep="--")

In [13]:
_spec_header = [
    r"\multicolumn{2}{l}{} & \multicolumn{3}{c}{Spectral Coherence RMSE} \\",
    r"\cmidrule(lr){3-5}",
    r"PDE & Model & Low & Mid & High \\",
]
print(to_latex_multiindex(
    t_spectral,
    caption=(
        r"Spectral coherence RMSE $\downarrow$ (averaged over rollout horizons) "
        r"for Low / Mid / High frequency bands."
    ),
    label="tab:appendix_spectral",
    header_lines=_spec_header,
))

\begin{table}[t]
\centering
\small
\begin{tabular}{llccc}
\toprule
\multicolumn{2}{l}{} & \multicolumn{3}{c}{Spectral Coherence RMSE} \\
\cmidrule(lr){3-5}
PDE & Model & Low & Mid & High \\
\midrule
\multirow{11}{*}{\rotatebox{90}{Turbulence}} & WFM-1 haar & 0.134 & 0.176 & \textbf{0.134} \\
 & WFM-1 db2 & 0.137 & 0.180 & 0.141 \\
 & WFM-1 db4 & \textbf{0.129} & 0.175 & 0.139 \\
 & WFM-1 db6 & 0.134 & 0.176 & 0.141 \\
 & WFM-3 haar & \underline{0.132} & \textbf{0.173} & 0.141 \\
 & WFM-3 db2 & 0.140 & 0.187 & \underline{0.136} \\
 & WFM-3 db4 & 0.141 & 0.178 & 0.144 \\
 & WFM-3 db6 & 0.137 & \underline{0.174} & 0.147 \\
 & FM$_\text{pixel}$ & 0.139 & 0.180 & 0.138 \\
 & FM$_\text{fourier}$ & 0.236 & 0.487 & 0.417 \\
 & WDNO & 0.199 & 0.249 & 0.365 \\
\midrule
\multirow{11}{*}{\rotatebox{90}{Rayleigh--B\'{e}nard}} & WFM-1 haar & 0.169 & 0.132 & 0.117 \\
 & WFM-1 db2 & 0.167 & 0.126 & 0.105 \\
 & WFM-1 db4 & 0.166 & 0.117 & \underline{0.077} \\
 & WFM-1 db6 & 0.188 & 0.125 & 0.085 \\
 & 

## A.3b  Spectral Coherence RMSE — by Band × Horizon (detailed)

Optional detailed breakdown: columns are `(band, horizon)` MultiIndex.

In [14]:
t_spec_hz = build_spectral_horizon_table(models=WFM_all + gen_models)
t_spec_hz.style.apply(highlight_min_pde, axis=None).format("{:.3f}", na_rep="--")

## A.4  Wasserstein-1 Distance — WFM + Generative Models

In [15]:
t_w1 = build_w1_table(models=WFM_all + gen_models)
t_w1.style.apply(highlight_min_pde, axis=None).format("{:.3f}", na_rep="--")

In [16]:
print(to_latex_multiindex(
    t_w1,
    caption=r"Wasserstein-1 distance $W_1$ $\downarrow$ for generative and WFM models.",
    label="tab:appendix_w1",
    header_lines=[r"PDE & Model & $W_1$ \\"],
))

\begin{table}[t]
\centering
\small
\begin{tabular}{llc}
\toprule
PDE & Model & $W_1$ \\
\midrule
\multirow{11}{*}{\rotatebox{90}{Turbulence}} & WFM-1 haar & 0.508 \\
 & WFM-1 db2 & 0.665 \\
 & WFM-1 db4 & 0.567 \\
 & WFM-1 db6 & \underline{0.493} \\
 & WFM-3 haar & 0.504 \\
 & WFM-3 db2 & 0.631 \\
 & WFM-3 db4 & \textbf{0.403} \\
 & WFM-3 db6 & 0.532 \\
 & FM$_\text{pixel}$ & 0.504 \\
 & FM$_\text{fourier}$ & 1.206 \\
 & WDNO & 3.530 \\
\midrule
\multirow{11}{*}{\rotatebox{90}{Rayleigh--B\'{e}nard}} & WFM-1 haar & \underline{0.018} \\
 & WFM-1 db2 & 0.019 \\
 & WFM-1 db4 & 0.025 \\
 & WFM-1 db6 & 0.037 \\
 & WFM-3 haar & 0.022 \\
 & WFM-3 db2 & 0.019 \\
 & WFM-3 db4 & \textbf{0.017} \\
 & WFM-3 db6 & 0.026 \\
 & FM$_\text{pixel}$ & 0.018 \\
 & FM$_\text{fourier}$ & 0.046 \\
 & WDNO & 0.110 \\
\midrule
\multirow{11}{*}{\rotatebox{90}{Active Matter}} & WFM-1 haar & 0.071 \\
 & WFM-1 db2 & 0.234 \\
 & WFM-1 db4 & 0.493 \\
 & WFM-1 db6 & 0.210 \\
 & WFM-3 haar & \underline{0.036} \\
 & WFM

---
# B. Ablation Tables

**Rows**: model display labels (WFM-1 variants, WFM-3 variants, generative baselines).
**Columns**: values of the physical parameter being swept.
Three metrics per parameter: **VRMSE** (all models), **CRPS** (generative + WFM), **Spectral RMSE** by band.

| Section | PDE              | Parameter   | New vs original |
|---------|------------------|-------------|-----------------|
| B.1     | Turbulence       | tcool       | + VRMSE (new)   |
| B.2     | Rayleigh–Bénard  | Ra          | + VRMSE (new)   |
| B.3     | Rayleigh–Bénard  | Pr          | + VRMSE (new)   |
| B.4     | Active Matter    | alpha       | + VRMSE (new)   |
| B.5     | Active Matter    | zeta        | + VRMSE (new)   |

## B.1  Turbulent Radiative Layer — $t_\mathrm{cool}$

In [17]:
_PDE  = "turbulent_radiative_layer_2D"
_PAR  = "tcool"
_FILT = lambda v: v <= 0.030001   # keep only the three standard tcool values
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"VRMSE $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.",
    label="tab:ablation_trl_vrmse_tcool",
))

tcool,0.030000
Model,
WFM-1 haar,0.669
WFM-1 db2,0.672
WFM-1 db4,0.664
WFM-1 db6,0.670
WFM-3 haar,0.659
WFM-3 db2,0.892
WFM-3 db4,0.678
WFM-3 db6,0.655
FM$_\text{pixel}$,0.668


\begin{table}[t]
\centering
\small
\begin{tabular}{lc}
\toprule
Model & 0.0299999993294477 \\
\midrule
WFM-1 haar & 0.669 \\
WFM-1 db2 & 0.672 \\
WFM-1 db4 & 0.664 \\
WFM-1 db6 & 0.670 \\
WFM-3 haar & 0.659 \\
WFM-3 db2 & 0.892 \\
WFM-3 db4 & 0.678 \\
WFM-3 db6 & \underline{0.655} \\
FM$_\text{pixel}$ & 0.668 \\
FM$_\text{fourier}$ & \textbf{0.651} \\
WDNO & 1.255 \\
\bottomrule
\end{tabular}
\caption{VRMSE $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.}
\label{tab:ablation_trl_vrmse_tcool}
\end{table}


In [18]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"CRPS $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.",
    label="tab:ablation_trl_crps_tcool",
))

tcool,0.030000
Model,
WFM-1 haar,1.122
WFM-1 db2,1.188
WFM-1 db4,1.126
WFM-1 db6,1.089
WFM-3 haar,1.085
WFM-3 db2,1.267
WFM-3 db4,1.031
WFM-3 db6,1.076
FM$_\text{pixel}$,1.161


\begin{table}[t]
\centering
\small
\begin{tabular}{lc}
\toprule
Model & 0.0299999993294477 \\
\midrule
WFM-1 haar & 1.122 \\
WFM-1 db2 & 1.188 \\
WFM-1 db4 & 1.126 \\
WFM-1 db6 & 1.089 \\
WFM-3 haar & 1.085 \\
WFM-3 db2 & 1.267 \\
WFM-3 db4 & \textbf{1.031} \\
WFM-3 db6 & \underline{1.076} \\
FM$_\text{pixel}$ & 1.161 \\
FM$_\text{fourier}$ & 1.390 \\
WDNO & 3.306 \\
\bottomrule
\end{tabular}
\caption{CRPS $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.}
\label{tab:ablation_trl_crps_tcool}
\end{table}


In [19]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.",
    label="tab:ablation_trl_spectral_tcool",
))

band,Low,Mid,High
tcool,0.030000,0.030000,0.030000
Model,,,
WFM-1 haar,0.137,0.191,0.142
WFM-1 db2,0.140,0.190,0.138
WFM-1 db4,0.131,0.188,0.134
WFM-1 db6,0.138,0.192,0.138
WFM-3 haar,0.132,0.189,0.152
WFM-3 db2,0.148,0.193,0.153
WFM-3 db4,0.149,0.192,0.146
WFM-3 db6,0.154,0.195,0.151


\begin{table}[t]
\centering
\small
\begin{tabular}{lccc}
\toprule
Model & ('Low', 0.0299999993294477) & ('Mid', 0.0299999993294477) & ('High', 0.0299999993294477) \\
\midrule
WFM-1 haar & 0.137 & 0.191 & 0.142 \\
WFM-1 db2 & 0.140 & 0.190 & \underline{0.138} \\
WFM-1 db4 & \textbf{0.131} & \textbf{0.188} & \textbf{0.134} \\
WFM-1 db6 & 0.138 & 0.192 & 0.138 \\
WFM-3 haar & \underline{0.132} & \underline{0.189} & 0.152 \\
WFM-3 db2 & 0.148 & 0.193 & 0.153 \\
WFM-3 db4 & 0.149 & 0.192 & 0.146 \\
WFM-3 db6 & 0.154 & 0.195 & 0.151 \\
FM$_\text{pixel}$ & 0.139 & 0.192 & 0.153 \\
FM$_\text{fourier}$ & 0.240 & 0.494 & 0.489 \\
WDNO & 0.205 & 0.254 & 0.360 \\
\bottomrule
\end{tabular}
\caption{Spectral coherence RMSE $\downarrow$ vs $t_{\mathrm{cool}}$ on Turbulence.}
\label{tab:ablation_trl_spectral_tcool}
\end{table}


## B.2  Rayleigh–Bénard — Rayleigh Number $Ra$

Prandtl number aggregated out (mean over all Pr values in the sweep).

In [20]:
_PDE = "rayleigh_benard"
_PAR = "Rayleigh"
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"VRMSE $\downarrow$ vs Rayleigh number $Ra$ on Rayleigh--B\'enard (averaged over $Pr$).",
    label="tab:ablation_rb_vrmse_ra",
))

Rayleigh,13.815511,18.420681,23.025850
Model,,,
WFM-1 haar,1.510,1.179,0.792
WFM-1 db2,1.414,1.003,0.769
WFM-1 db4,1.758,1.080,0.782
WFM-1 db6,1.512,1.129,1.033
WFM-3 haar,1.793,0.989,0.775
WFM-3 db2,1.307,0.828,0.761
WFM-3 db4,1.498,0.767,0.714
WFM-3 db6,1.314,0.761,0.770
FM$_\text{pixel}$,1.352,1.084,0.792


\begin{table}[t]
\centering
\small
\begin{tabular}{lccc}
\toprule
Model & 13.815510749816896 & 18.42068099975586 & 23.025850296020508 \\
\midrule
WFM-1 haar & 1.510 & 1.179 & 0.792 \\
WFM-1 db2 & 1.414 & 1.003 & 0.769 \\
WFM-1 db4 & 1.758 & 1.080 & 0.782 \\
WFM-1 db6 & 1.512 & 1.129 & 1.033 \\
WFM-3 haar & 1.793 & 0.989 & 0.775 \\
WFM-3 db2 & \underline{1.307} & 0.828 & \underline{0.761} \\
WFM-3 db4 & 1.498 & \underline{0.767} & \textbf{0.714} \\
WFM-3 db6 & 1.314 & \textbf{0.761} & 0.770 \\
FM$_\text{pixel}$ & 1.352 & 1.084 & 0.792 \\
FM$_\text{fourier}$ & \textbf{0.795} & 1.036 & 0.798 \\
WDNO & 8.640 & 5.080 & 2.604 \\
\bottomrule
\end{tabular}
\caption{VRMSE $\downarrow$ vs Rayleigh number $Ra$ on Rayleigh--B\'enard (averaged over $Pr$).}
\label{tab:ablation_rb_vrmse_ra}
\end{table}


In [21]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"CRPS $\downarrow$ vs $Ra$ on Rayleigh--B\'enard (averaged over $Pr$).",
    label="tab:ablation_rb_crps_ra",
))

Rayleigh,13.815511,18.420681,23.025850
Model,,,
WFM-1 haar,0.058,0.065,0.054
WFM-1 db2,0.058,0.057,0.052
WFM-1 db4,0.058,0.059,0.052
WFM-1 db6,0.062,0.068,0.076
WFM-3 haar,0.059,0.063,0.054
WFM-3 db2,0.058,0.057,0.053
WFM-3 db4,0.054,0.057,0.048
WFM-3 db6,0.059,0.056,0.055
FM$_\text{pixel}$,0.056,0.066,0.055


\begin{table}[t]
\centering
\small
\begin{tabular}{lccc}
\toprule
Model & 13.815510749816896 & 18.42068099975586 & 23.025850296020508 \\
\midrule
WFM-1 haar & 0.058 & 0.065 & 0.054 \\
WFM-1 db2 & 0.058 & \underline{0.057} & \underline{0.052} \\
WFM-1 db4 & 0.058 & 0.059 & 0.052 \\
WFM-1 db6 & 0.062 & 0.068 & 0.076 \\
WFM-3 haar & 0.059 & 0.063 & 0.054 \\
WFM-3 db2 & 0.058 & 0.057 & 0.053 \\
WFM-3 db4 & \textbf{0.054} & 0.057 & \textbf{0.048} \\
WFM-3 db6 & 0.059 & \textbf{0.056} & 0.055 \\
FM$_\text{pixel}$ & \underline{0.056} & 0.066 & 0.055 \\
FM$_\text{fourier}$ & 0.071 & 0.081 & 0.064 \\
WDNO & 0.120 & 0.118 & 0.116 \\
\bottomrule
\end{tabular}
\caption{CRPS $\downarrow$ vs $Ra$ on Rayleigh--B\'enard (averaged over $Pr$).}
\label{tab:ablation_rb_crps_ra}
\end{table}


In [22]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $Ra$ on Rayleigh--B\'enard (averaged over $Pr$).",
    label="tab:ablation_rb_spectral_ra",
))

\begin{table}[t]
\centering
\small
\begin{tabular}{lccccccccc}
\toprule
Model & ('Low', 13.815510749816896) & ('Low', 18.42068099975586) & ('Low', 23.025850296020508) & ('Mid', 13.815510749816896) & ('Mid', 18.42068099975586) & ('Mid', 23.025850296020508) & ('High', 13.815510749816896) & ('High', 18.42068099975586) & ('High', 23.025850296020508) \\
\midrule
WFM-1 haar & 0.172 & 0.158 & 0.135 & 0.189 & 0.136 & 0.132 & 0.112 & 0.188 & 0.163 \\
WFM-1 db2 & 0.163 & \underline{0.157} & 0.143 & 0.165 & 0.131 & 0.133 & 0.094 & 0.163 & 0.151 \\
WFM-1 db4 & 0.150 & 0.169 & 0.141 & \textbf{0.116} & 0.130 & 0.132 & \textbf{0.048} & 0.102 & \textbf{0.120} \\
WFM-1 db6 & 0.184 & 0.190 & 0.209 & 0.141 & 0.125 & 0.177 & 0.053 & \textbf{0.090} & 0.181 \\
WFM-3 haar & 0.157 & 0.167 & \underline{0.129} & 0.186 & 0.133 & 0.129 & 0.108 & 0.181 & 0.158 \\
WFM-3 db2 & \underline{0.144} & 0.159 & \textbf{0.127} & 0.134 & \underline{0.120} & \textbf{0.125} & 0.060 & 0.106 & \underline{0.120} \\
WFM-3 db4 & \t

## B.3  Rayleigh–Bénard — Prandtl Number $Pr$

Rayleigh number aggregated out (mean over all Ra values in the sweep).

In [23]:
_PDE = "rayleigh_benard"
_PAR = "Prandtl"
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"VRMSE $\downarrow$ vs $Pr$ on Rayleigh--B\'enard (averaged over $Ra$).",
    label="tab:ablation_rb_vrmse_pr",
))

Prandtl,-2.302585,0.000000,2.302585
Model,,,
WFM-1 haar,1.491,0.921,1.070
WFM-1 db2,1.195,0.886,1.106
WFM-1 db4,1.344,1.070,1.207
WFM-1 db6,1.315,1.104,1.256
WFM-3 haar,1.410,0.921,1.226
WFM-3 db2,1.115,0.866,0.915
WFM-3 db4,1.144,0.873,0.961
WFM-3 db6,0.917,0.836,1.091
FM$_\text{pixel}$,1.188,0.953,1.088


\begin{table}[t]
\centering
\small
\begin{tabular}{lccc}
\toprule
Model & -2.3025851249694824 & 0.0 & 2.3025851249694824 \\
\midrule
WFM-1 haar & 1.491 & 0.921 & 1.070 \\
WFM-1 db2 & 1.195 & 0.886 & 1.106 \\
WFM-1 db4 & 1.344 & 1.070 & 1.207 \\
WFM-1 db6 & 1.315 & 1.104 & 1.256 \\
WFM-3 haar & 1.410 & 0.921 & 1.226 \\
WFM-3 db2 & 1.115 & 0.866 & \underline{0.915} \\
WFM-3 db4 & 1.144 & 0.873 & 0.961 \\
WFM-3 db6 & \textbf{0.917} & \underline{0.836} & 1.091 \\
FM$_\text{pixel}$ & 1.188 & 0.953 & 1.088 \\
FM$_\text{fourier}$ & \underline{1.072} & \textbf{0.762} & \textbf{0.796} \\
WDNO & 4.491 & 3.943 & 7.889 \\
\bottomrule
\end{tabular}
\caption{VRMSE $\downarrow$ vs $Pr$ on Rayleigh--B\'enard (averaged over $Ra$).}
\label{tab:ablation_rb_vrmse_pr}
\end{table}


In [24]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"CRPS $\downarrow$ vs $Pr$ on Rayleigh--B\'enard (averaged over $Ra$).",
    label="tab:ablation_rb_crps_pr",
))

Prandtl,-2.302585,0.000000,2.302585
Model,,,
WFM-1 haar,0.094,0.054,0.029
WFM-1 db2,0.083,0.052,0.031
WFM-1 db4,0.085,0.050,0.034
WFM-1 db6,0.093,0.065,0.049
WFM-3 haar,0.095,0.049,0.032
WFM-3 db2,0.086,0.051,0.031
WFM-3 db4,0.082,0.049,0.028
WFM-3 db6,0.083,0.048,0.039
FM$_\text{pixel}$,0.090,0.054,0.033


\begin{table}[t]
\centering
\small
\begin{tabular}{lccc}
\toprule
Model & -2.3025851249694824 & 0.0 & 2.3025851249694824 \\
\midrule
WFM-1 haar & 0.094 & 0.054 & \underline{0.029} \\
WFM-1 db2 & \underline{0.083} & 0.052 & 0.031 \\
WFM-1 db4 & 0.085 & 0.050 & 0.034 \\
WFM-1 db6 & 0.093 & 0.065 & 0.049 \\
WFM-3 haar & 0.095 & \underline{0.049} & 0.032 \\
WFM-3 db2 & 0.086 & 0.051 & 0.031 \\
WFM-3 db4 & \textbf{0.082} & 0.049 & \textbf{0.028} \\
WFM-3 db6 & 0.083 & \textbf{0.048} & 0.039 \\
FM$_\text{pixel}$ & 0.090 & 0.054 & 0.033 \\
FM$_\text{fourier}$ & 0.114 & 0.058 & 0.044 \\
WDNO & 0.145 & 0.112 & 0.097 \\
\bottomrule
\end{tabular}
\caption{CRPS $\downarrow$ vs $Pr$ on Rayleigh--B\'enard (averaged over $Ra$).}
\label{tab:ablation_rb_crps_pr}
\end{table}


In [25]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $Pr$ on Rayleigh--B\'enard (averaged over $Ra$).",
    label="tab:ablation_rb_spectral_pr",
))

\begin{table}[t]
\centering
\small
\begin{tabular}{lccccccccc}
\toprule
Model & ('Low', -2.3025851249694824) & ('Low', 0.0) & ('Low', 2.3025851249694824) & ('Mid', -2.3025851249694824) & ('Mid', 0.0) & ('Mid', 2.3025851249694824) & ('High', -2.3025851249694824) & ('High', 0.0) & ('High', 2.3025851249694824) \\
\midrule
WFM-1 haar & 0.175 & 0.142 & 0.147 & 0.159 & 0.165 & 0.133 & 0.193 & 0.160 & 0.110 \\
WFM-1 db2 & 0.169 & 0.147 & 0.146 & 0.151 & 0.155 & 0.124 & 0.182 & 0.138 & 0.089 \\
WFM-1 db4 & 0.178 & 0.138 & 0.144 & 0.138 & \textbf{0.130} & 0.111 & 0.121 & \textbf{0.086} & \textbf{0.063} \\
WFM-1 db6 & 0.212 & 0.183 & 0.188 & 0.151 & 0.151 & 0.142 & 0.130 & 0.118 & 0.075 \\
WFM-3 haar & 0.182 & 0.136 & 0.134 & 0.156 & 0.157 & 0.135 & 0.189 & 0.150 & 0.108 \\
WFM-3 db2 & 0.167 & \textbf{0.133} & \underline{0.130} & \underline{0.129} & 0.134 & 0.115 & 0.123 & 0.096 & \underline{0.067} \\
WFM-3 db4 & \textbf{0.159} & \underline{0.133} & \textbf{0.129} & \textbf{0.126} & \underline{0

## B.4  Active Matter — $\alpha$

Zeta aggregated out. Filtered to the three main sweep values: $\alpha \in \{-3, -4, -5\}$.

In [26]:
_PDE  = "active_matter"
_PAR  = "alpha"
_FILT = lambda v: v in [-3, -4, -5]
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"VRMSE $\downarrow$ vs $\alpha$ on Active Matter (averaged over $\zeta$).",
    label="tab:ablation_am_vrmse_alpha",
))

alpha,-5.000000,-4.000000,-3.000000
Model,,,
WFM-1 haar,0.957,0.960,0.974
WFM-1 db2,1.262,1.267,1.308
WFM-1 db4,1.844,1.909,1.997
WFM-1 db6,1.201,1.219,1.272
WFM-3 haar,0.914,0.913,0.902
WFM-3 db2,0.936,0.945,0.938
WFM-3 db4,1.133,1.212,1.165
WFM-3 db6,1.070,1.057,1.039
FM$_\text{pixel}$,0.877,0.874,0.861


\begin{table}[t]
\centering
\small
\begin{tabular}{lccc}
\toprule
Model & -5.0 & -4.0 & -3.0 \\
\midrule
WFM-1 haar & 0.957 & 0.960 & 0.974 \\
WFM-1 db2 & 1.262 & 1.267 & 1.308 \\
WFM-1 db4 & 1.844 & 1.909 & 1.997 \\
WFM-1 db6 & 1.201 & 1.219 & 1.272 \\
WFM-3 haar & \underline{0.914} & \underline{0.913} & \underline{0.902} \\
WFM-3 db2 & 0.936 & 0.945 & 0.938 \\
WFM-3 db4 & 1.133 & 1.212 & 1.165 \\
WFM-3 db6 & 1.070 & 1.057 & 1.039 \\
FM$_\text{pixel}$ & \textbf{0.877} & \textbf{0.874} & \textbf{0.861} \\
FM$_\text{fourier}$ & 0.958 & 0.998 & 1.034 \\
WDNO & 1.188 & 1.209 & 1.221 \\
\bottomrule
\end{tabular}
\caption{VRMSE $\downarrow$ vs $\alpha$ on Active Matter (averaged over $\zeta$).}
\label{tab:ablation_am_vrmse_alpha}
\end{table}


In [27]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"CRPS $\downarrow$ vs $\alpha$ on Active Matter (averaged over $\zeta$).",
    label="tab:ablation_am_crps_alpha",
))

alpha,-5.000000,-4.000000,-3.000000
Model,,,
WFM-1 haar,0.290,0.225,0.191
WFM-1 db2,0.339,0.273,0.244
WFM-1 db4,0.443,0.393,0.377
WFM-1 db6,0.304,0.248,0.225
WFM-3 haar,0.281,0.217,0.175
WFM-3 db2,0.284,0.221,0.180
WFM-3 db4,0.316,0.265,0.214
WFM-3 db6,0.296,0.232,0.193
FM$_\text{pixel}$,0.272,0.209,0.169


\begin{table}[t]
\centering
\small
\begin{tabular}{lccc}
\toprule
Model & -5.0 & -4.0 & -3.0 \\
\midrule
WFM-1 haar & 0.290 & 0.225 & 0.191 \\
WFM-1 db2 & 0.339 & 0.273 & 0.244 \\
WFM-1 db4 & 0.443 & 0.393 & 0.377 \\
WFM-1 db6 & 0.304 & 0.248 & 0.225 \\
WFM-3 haar & \underline{0.281} & \underline{0.217} & \underline{0.175} \\
WFM-3 db2 & 0.284 & 0.221 & 0.180 \\
WFM-3 db4 & 0.316 & 0.265 & 0.214 \\
WFM-3 db6 & 0.296 & 0.232 & 0.193 \\
FM$_\text{pixel}$ & \textbf{0.272} & \textbf{0.209} & \textbf{0.169} \\
FM$_\text{fourier}$ & 0.318 & 0.260 & 0.231 \\
WDNO & 0.382 & 0.312 & 0.267 \\
\bottomrule
\end{tabular}
\caption{CRPS $\downarrow$ vs $\alpha$ on Active Matter (averaged over $\zeta$).}
\label{tab:ablation_am_crps_alpha}
\end{table}


In [28]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR, param_filter=_FILT)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $\alpha$ on Active Matter (averaged over $\zeta$).",
    label="tab:ablation_am_spectral_alpha",
))

\begin{table}[t]
\centering
\small
\begin{tabular}{lccccccccc}
\toprule
Model & ('Low', -5.0) & ('Low', -4.0) & ('Low', -3.0) & ('Mid', -5.0) & ('Mid', -4.0) & ('Mid', -3.0) & ('High', -5.0) & ('High', -4.0) & ('High', -3.0) \\
\midrule
WFM-1 haar & 0.187 & 0.186 & 0.201 & \textbf{0.236} & 0.224 & 0.253 & \underline{0.202} & \underline{0.244} & \underline{0.232} \\
WFM-1 db2 & 0.254 & 0.245 & 0.266 & 0.285 & 0.279 & 0.310 & 0.222 & 0.312 & 0.281 \\
WFM-1 db4 & 0.256 & 0.253 & 0.273 & 0.299 & 0.298 & 0.321 & 0.226 & 0.325 & 0.292 \\
WFM-1 db6 & 0.226 & 0.225 & 0.253 & 0.297 & 0.296 & 0.323 & 0.223 & 0.314 & 0.284 \\
WFM-3 haar & 0.190 & 0.185 & 0.194 & 0.255 & 0.229 & 0.254 & 0.311 & 0.318 & 0.302 \\
WFM-3 db2 & \underline{0.186} & 0.181 & 0.184 & \underline{0.237} & \underline{0.217} & 0.243 & 0.232 & 0.259 & 0.234 \\
WFM-3 db4 & 0.219 & 0.212 & 0.208 & 0.302 & 0.268 & 0.281 & 0.258 & 0.311 & 0.282 \\
WFM-3 db6 & \textbf{0.180} & \textbf{0.177} & \underline{0.181} & 0.248 & 0.220 & \un

## B.5  Active Matter — $\zeta$

Alpha aggregated out (mean over all alpha values in the sweep).

In [29]:
_PDE = "active_matter"
_PAR = "zeta"
_pde_label = PDE_LABELS[_PDE]

# -- VRMSE --------------------------------------------------------------------
t = build_ablation_table("vrmse", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"VRMSE $\downarrow$ vs $\zeta$ on Active Matter (averaged over $\alpha$).",
    label="tab:ablation_am_vrmse_zeta",
))

zeta,1.000000,5.000000,11.000000,17.000000
Model,,,,
WFM-1 haar,4.098,0.961,0.993,0.969
WFM-1 db2,8.845,1.351,1.427,1.300
WFM-1 db4,21.020,2.060,2.263,1.974
WFM-1 db6,10.165,1.286,1.382,1.279
WFM-3 haar,1.811,0.877,0.860,0.916
WFM-3 db2,1.885,0.907,0.938,0.951
WFM-3 db4,5.455,1.140,1.213,1.209
WFM-3 db6,4.347,1.048,1.074,1.066
FM$_\text{pixel}$,1.828,0.830,0.823,0.872


\begin{table}[t]
\centering
\small
\begin{tabular}{lcccc}
\toprule
Model & 1.0 & 5.0 & 11.0 & 17.0 \\
\midrule
WFM-1 haar & 4.098 & 0.961 & 0.993 & 0.969 \\
WFM-1 db2 & 8.845 & 1.351 & 1.427 & 1.300 \\
WFM-1 db4 & 21.020 & 2.060 & 2.263 & 1.974 \\
WFM-1 db6 & 10.165 & 1.286 & 1.382 & 1.279 \\
WFM-3 haar & \underline{1.811} & \underline{0.877} & \underline{0.860} & \underline{0.916} \\
WFM-3 db2 & 1.885 & 0.907 & 0.938 & 0.951 \\
WFM-3 db4 & 5.455 & 1.140 & 1.213 & 1.209 \\
WFM-3 db6 & 4.347 & 1.048 & 1.074 & 1.066 \\
FM$_\text{pixel}$ & 1.828 & \textbf{0.830} & \textbf{0.823} & \textbf{0.872} \\
FM$_\text{fourier}$ & \textbf{1.474} & 0.987 & 1.042 & 1.018 \\
WDNO & 6.435 & 1.221 & 1.302 & 1.246 \\
\bottomrule
\end{tabular}
\caption{VRMSE $\downarrow$ vs $\zeta$ on Active Matter (averaged over $\alpha$).}
\label{tab:ablation_am_vrmse_zeta}
\end{table}


In [30]:
# -- CRPS ---------------------------------------------------------------------
t = build_ablation_table("crps", _PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"CRPS $\downarrow$ vs $\zeta$ on Active Matter (averaged over $\alpha$).",
    label="tab:ablation_am_crps_zeta",
))

zeta,1.000000,5.000000,11.000000,17.000000
Model,,,,
WFM-1 haar,0.113,0.216,0.173,0.200
WFM-1 db2,0.187,0.271,0.232,0.252
WFM-1 db4,0.350,0.398,0.369,0.376
WFM-1 db6,0.174,0.246,0.213,0.231
WFM-3 haar,0.097,0.200,0.153,0.193
WFM-3 db2,0.096,0.202,0.164,0.197
WFM-3 db4,0.137,0.238,0.199,0.232
WFM-3 db6,0.134,0.219,0.178,0.208
FM$_\text{pixel}$,0.104,0.191,0.151,0.186


\begin{table}[t]
\centering
\small
\begin{tabular}{lcccc}
\toprule
Model & 1.0 & 5.0 & 11.0 & 17.0 \\
\midrule
WFM-1 haar & 0.113 & 0.216 & 0.173 & 0.200 \\
WFM-1 db2 & 0.187 & 0.271 & 0.232 & 0.252 \\
WFM-1 db4 & 0.350 & 0.398 & 0.369 & 0.376 \\
WFM-1 db6 & 0.174 & 0.246 & 0.213 & 0.231 \\
WFM-3 haar & \underline{0.097} & \underline{0.200} & \underline{0.153} & \underline{0.193} \\
WFM-3 db2 & \textbf{0.096} & 0.202 & 0.164 & 0.197 \\
WFM-3 db4 & 0.137 & 0.238 & 0.199 & 0.232 \\
WFM-3 db6 & 0.134 & 0.219 & 0.178 & 0.208 \\
FM$_\text{pixel}$ & 0.104 & \textbf{0.191} & \textbf{0.151} & \textbf{0.186} \\
FM$_\text{fourier}$ & 0.132 & 0.247 & 0.211 & 0.238 \\
WDNO & 0.164 & 0.292 & 0.249 & 0.283 \\
\bottomrule
\end{tabular}
\caption{CRPS $\downarrow$ vs $\zeta$ on Active Matter (averaged over $\alpha$).}
\label{tab:ablation_am_crps_zeta}
\end{table}


In [31]:
# -- Spectral RMSE ------------------------------------------------------------
t = build_spectral_ablation_table(_PDE, _PAR)
display(t.style.apply(highlight_min, axis=None).format("{:.3f}", na_rep="--"))
print(to_latex_simple(
    t,
    caption=r"Spectral coherence RMSE $\downarrow$ vs $\zeta$ on Active Matter (averaged over $\alpha$).",
    label="tab:ablation_am_spectral_zeta",
))

\begin{table}[t]
\centering
\small
\begin{tabular}{lcccccccccccc}
\toprule
Model & ('Low', 1.0) & ('Low', 5.0) & ('Low', 11.0) & ('Low', 17.0) & ('Mid', 1.0) & ('Mid', 5.0) & ('Mid', 11.0) & ('Mid', 17.0) & ('High', 1.0) & ('High', 5.0) & ('High', 11.0) & ('High', 17.0) \\
\midrule
WFM-1 haar & 0.339 & 0.190 & 0.231 & 0.186 & 0.421 & 0.249 & 0.284 & 0.223 & 0.106 & \underline{0.196} & 0.238 & \underline{0.257} \\
WFM-1 db2 & 0.387 & 0.263 & 0.297 & 0.242 & 0.458 & 0.302 & 0.336 & 0.276 & 0.122 & 0.212 & 0.288 & 0.333 \\
WFM-1 db4 & 0.387 & 0.266 & 0.303 & 0.252 & 0.451 & 0.309 & 0.348 & 0.294 & 0.131 & 0.216 & 0.304 & 0.342 \\
WFM-1 db6 & 0.379 & 0.242 & 0.283 & 0.231 & 0.462 & 0.312 & 0.350 & 0.294 & 0.125 & 0.211 & 0.289 & 0.335 \\
WFM-3 haar & 0.341 & 0.183 & 0.196 & 0.185 & 0.443 & 0.263 & 0.265 & 0.223 & 0.112 & 0.281 & 0.300 & 0.320 \\
WFM-3 db2 & 0.328 & 0.175 & 0.196 & \underline{0.173} & \underline{0.408} & \underline{0.246} & 0.264 & 0.215 & \textbf{0.074} & 0.199 & \underlin